# graph_adapter_batch_v1

验证 graph/network family adapter：`market_graph_mse`、`gcn_mse`、`gat_mse`、`rgcn_mse`、`rsr_mse`、`hypergraph_mse`、`thgnn_mse`、`sthan_mse`、`sthgcn_mse`。本轮目标是把 graph propagation、attention、relation、hypergraph、temporal graph 特征 proxy 接入统一 `src/model` 训练、预测、artifact 和 matrix analysis 链路。

## 实验治理

- Baseline：graph family 之前仍落在 smoke fallback。
- Direct comparison：static graph、attention graph、relation graph、hypergraph、temporal graph proxy 是否输出不同预测并保持同一 artifact/manifest 契约。
- Role：diagnose，推进 QuantBench graph/network family 从 smoke fallback 迁移到可训练 adapter。
- Target window：2026-01-01 到 2026-06-09，2026-04-30 前训练，之后测试。
- Changed layer：model/network。
- External reference：GCN、GAT、R-GCN、Hypergraph NN 以及时空图/超图模型方向。当前是可训练 graph-feature proxy，不宣称完整论文级 GNN。
- Acceptance：九个模型输出 `paper_strict_adapter`；attention/relation/hypergraph/temporal graph 路径互不退化为同一个 fallback；GAT artifact 保存/加载后保留 adapter metadata。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').exists() and (path / 'AGENTS.md').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact, load_model_artifact, save_model_artifact
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_predict_model, model_adapter_name, model_adapter_status

CARD = repo_root / 'research' / 'graph_adapter_batch_v1'
CARD.mkdir(parents=True, exist_ok=True)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
test = dataset.frame.loc[dataset.frame['date'] > '2026-04-30'].copy()
feature_columns = dataset.feature_columns[:8]
{
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'train_end': str(train['date'].max().date()),
    'test_start': str(test['date'].min().date()),
    'test_end': str(test['date'].max().date()),
}


{'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level',
  'vix_change_1d',
  'intraday_range'],
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08'}

In [2]:
model_names = (
    'market_graph_mse',
    'gcn_mse',
    'gat_mse',
    'rgcn_mse',
    'rsr_mse',
    'hypergraph_mse',
    'thgnn_mse',
    'sthan_mse',
    'sthgcn_mse',
)
expected = {
    'market_graph_mse': ('paper_strict_adapter', 'market_graph_feature_mse_v1'),
    'gcn_mse': ('paper_strict_adapter', 'gcn_feature_propagation_mse_v1'),
    'gat_mse': ('paper_strict_adapter', 'gat_feature_attention_mse_v1'),
    'rgcn_mse': ('paper_strict_adapter', 'rgcn_relation_feature_mse_v1'),
    'rsr_mse': ('paper_strict_adapter', 'rsr_relation_strength_mse_v1'),
    'hypergraph_mse': ('paper_strict_adapter', 'hypergraph_feature_mse_v1'),
    'thgnn_mse': ('paper_strict_adapter', 'thgnn_temporal_hypergraph_mse_v1'),
    'sthan_mse': ('paper_strict_adapter', 'sthan_spatio_temporal_attention_mse_v1'),
    'sthgcn_mse': ('paper_strict_adapter', 'sthgcn_spatio_temporal_hypergraph_mse_v1'),
}

predictions = pd.concat(
    [fit_predict_model(name, train=train, test=test, feature_columns=feature_columns) for name in model_names],
    ignore_index=True,
)
predictions.to_csv(CARD / 'adapter_predictions.csv', index=False)

for name, (status, adapter_name) in expected.items():
    subset = predictions.loc[predictions['model'] == name]
    assert subset['adapter_status'].unique().tolist() == [status]
    assert subset['model_adapter'].unique().tolist() == [adapter_name]
    assert model_adapter_status(name) == status
    assert model_adapter_name(name) == adapter_name

wide = predictions.pivot(index='date', columns='model', values='prediction').reset_index()
wide['gat_minus_gcn'] = wide['gat_mse'] - wide['gcn_mse']
wide['rgcn_minus_gcn'] = wide['rgcn_mse'] - wide['gcn_mse']
wide['hypergraph_minus_gcn'] = wide['hypergraph_mse'] - wide['gcn_mse']
wide['thgnn_minus_hypergraph'] = wide['thgnn_mse'] - wide['hypergraph_mse']
wide['sthan_minus_hypergraph'] = wide['sthan_mse'] - wide['hypergraph_mse']
wide['sthgcn_minus_hypergraph'] = wide['sthgcn_mse'] - wide['hypergraph_mse']
wide.to_csv(CARD / 'graph_prediction_deltas.csv', index=False)

assert not wide['gat_mse'].round(10).equals(wide['gcn_mse'].round(10))
assert not wide['rgcn_mse'].round(10).equals(wide['gcn_mse'].round(10))
assert not wide['hypergraph_mse'].round(10).equals(wide['gcn_mse'].round(10))
assert not wide['thgnn_mse'].round(10).equals(wide['hypergraph_mse'].round(10))
wide.head()


model,date,gat_mse,gcn_mse,hypergraph_mse,market_graph_mse,rgcn_mse,rsr_mse,sthan_mse,sthgcn_mse,thgnn_mse,gat_minus_gcn,rgcn_minus_gcn,hypergraph_minus_gcn,thgnn_minus_hypergraph,sthan_minus_hypergraph,sthgcn_minus_hypergraph
0,2026-05-01,-0.000616,0.005933,-0.009563,0.005909,0.005987,0.005030,-0.010240,-0.010480,-0.010517,-0.006548,0.000054,-0.015496,-0.000954,-0.000677,-0.000917
1,2026-05-04,0.004614,0.012122,-0.003527,0.012081,0.012212,0.010362,0.027130,0.028923,0.028433,-0.007508,0.000091,-0.015649,0.031960,0.030657,0.032450
2,2026-05-05,0.025079,0.032520,0.027032,0.032524,0.032570,0.033535,0.029163,0.047971,0.047884,-0.007441,0.000050,-0.005487,0.020852,0.002131,0.020938
3,2026-05-06,0.036690,0.048783,0.039500,0.048786,0.048960,0.050121,0.009950,0.026370,0.026429,-0.012093,0.000177,-0.009283,-0.013072,-0.029550,-0.013131
4,2026-05-07,-0.004259,0.001379,0.028160,0.001331,0.001284,-0.001067,-0.013399,0.003380,0.003526,-0.005638,-0.000095,0.026780,-0.024634,-0.041559,-0.024779


In [3]:
summary_rows = []
group_columns = ['model', 'objective', 'family', 'adapter_status', 'model_adapter']
for keys, group in predictions.groupby(group_columns, dropna=False):
    summary_rows.append(
        {
            **dict(zip(group_columns, keys)),
            'rows': int(len(group)),
            'prediction_mean': float(group['prediction'].mean()),
            'prediction_std': float(group['prediction'].std(ddof=0)),
            'target_corr': float(group['prediction'].corr(group['target_return'])),
        }
    )
adapter_summary = pd.DataFrame(summary_rows)
adapter_summary.to_csv(CARD / 'adapter_summary.csv', index=False)
adapter_summary


,model,objective,family,adapter_status,model_adapter,rows,prediction_mean,prediction_std,target_corr
0,gat_mse,mse,graph,paper_strict_adapter,gat_feature_attention_mse_v1,26,0.001011,0.024515,0.313457
1,gcn_mse,mse,graph,paper_strict_adapter,gcn_feature_propagation_mse_v1,26,0.018838,0.017486,0.011636
2,hypergraph_mse,mse,graph,paper_strict_adapter,hypergraph_feature_mse_v1,26,0.023392,0.019069,-0.111612
3,market_graph_mse,mse,graph,paper_strict_adapter,market_graph_feature_mse_v1,26,0.018805,0.017496,0.011297
4,rgcn_mse,mse,graph,paper_strict_adapter,rgcn_relation_feature_mse_v1,26,0.018901,0.017550,0.012688
5,rsr_mse,mse,graph,paper_strict_adapter,rsr_relation_strength_mse_v1,26,0.017826,0.018472,-0.003111
6,sthan_mse,mse,graph,paper_strict_adapter,sthan_spatio_temporal_attention_mse_v1,26,0.006385,0.046419,0.286848
7,sthgcn_mse,mse,graph,paper_strict_adapter,sthgcn_spatio_temporal_hypergraph_mse_v1,26,0.027783,0.033284,-0.012576
8,thgnn_mse,mse,graph,paper_strict_adapter,thgnn_temporal_hypergraph_mse_v1,26,0.027713,0.033094,-0.014624


In [4]:
artifact = fit_model_artifact('gat_mse', train=train, feature_columns=feature_columns)
artifact_path = save_model_artifact(CARD / 'gat_mse_artifact.joblib', artifact)
loaded = load_model_artifact(artifact_path)
artifact_predictions = loaded.predict(test)
artifact_metadata = loaded.metadata_payload()
artifact_predictions.to_csv(CARD / 'gat_artifact_predictions.csv', index=False)
(CARD / 'gat_artifact_metadata.json').write_text(
    json.dumps(artifact_metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

assert loaded.adapter_status == 'paper_strict_adapter'
assert artifact_metadata['model_adapter'] == 'gat_feature_attention_mse_v1'
assert artifact_predictions['model_adapter'].unique().tolist() == ['gat_feature_attention_mse_v1']
{
    'artifact_path': str(artifact_path.relative_to(CARD)),
    'metadata': artifact_metadata,
    'prediction_rows': int(len(artifact_predictions)),
}


{'artifact_path': 'gat_mse_artifact.joblib',
 'metadata': {'schema_version': 'model_artifact.v1',
  'model_name': 'gat_mse',
  'feature_columns': ['soxl_return_1d',
   'soxx_return_1d',
   'soxl_return_5d',
   'soxx_return_5d',
   'soxl_soxx_relative_5d',
   'vix_level',
   'vix_change_1d',
   'intraday_range'],
  'objective': 'mse',
  'family': 'graph',
  'adapter_status': 'paper_strict_adapter',
  'model_adapter': 'gat_feature_attention_mse_v1',
  'target_column': 'target_return',
  'train_rows': 82},
 'prediction_rows': 26}

In [5]:
matrix = run_quantbench_matrix_smoke(
    card_dir=CARD / 'matrix_card',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=model_names,
)
manifest = json.loads((CARD / 'matrix_card' / 'artifact_manifest.json').read_text(encoding='utf-8'))
assert manifest['metadata']['adapter_status'] == 'paper_strict_adapter'
assert manifest['metadata']['adapter_statuses'] == ['paper_strict_adapter']
matrix.model_comparison.to_csv(CARD / 'matrix_model_comparison.csv', index=False)
{
    'matrix_status': matrix.summary['status'],
    'top_model': matrix.summary['top_model'],
    'adapter_status': manifest['metadata']['adapter_status'],
    'model_adapters': manifest['metadata']['model_adapters'],
    'manifest_hash': manifest['manifest_hash'],
}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'matrix_status': 'smoke_completed',
 'top_model': 'gat_mse',
 'adapter_status': 'paper_strict_adapter',
 'model_adapters': ['gat_feature_attention_mse_v1',
  'gcn_feature_propagation_mse_v1',
  'hypergraph_feature_mse_v1',
  'market_graph_feature_mse_v1',
  'rgcn_relation_feature_mse_v1',
  'rsr_relation_strength_mse_v1',
  'sthan_spatio_temporal_attention_mse_v1',
  'sthgcn_spatio_temporal_hypergraph_mse_v1',
  'thgnn_temporal_hypergraph_mse_v1'],
 'manifest_hash': '403cbf18d3997d20e91eca465b1e03a9934f12200cc0e30c5b820c89a291d845'}

In [6]:
analysis_summary = {
    'schema_version': 'graph_adapter_batch_review.v1',
    'status': 'completed',
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'models': list(model_names),
    'adapter_statuses': sorted(predictions['adapter_status'].unique().tolist()),
    'model_adapters': sorted(predictions['model_adapter'].unique().tolist()),
    'gat_vs_gcn_abs_mean_diff': float(wide['gat_minus_gcn'].abs().mean()),
    'rgcn_vs_gcn_abs_mean_diff': float(wide['rgcn_minus_gcn'].abs().mean()),
    'hypergraph_vs_gcn_abs_mean_diff': float(wide['hypergraph_minus_gcn'].abs().mean()),
    'thgnn_vs_hypergraph_abs_mean_diff': float(wide['thgnn_minus_hypergraph'].abs().mean()),
    'adapter_summary_path': 'adapter_summary.csv',
    'prediction_path': 'adapter_predictions.csv',
    'delta_path': 'graph_prediction_deltas.csv',
    'artifact_path': 'gat_mse_artifact.joblib',
    'matrix_manifest': 'matrix_card/artifact_manifest.json',
    'matrix_top_model': matrix.summary['top_model'],
    'matrix_manifest_hash': manifest['manifest_hash'],
    'external_references': [
        'https://arxiv.org/abs/1609.02907',
        'https://arxiv.org/abs/1710.10903',
        'https://arxiv.org/abs/1703.06103',
        'https://arxiv.org/abs/1809.09401',
    ],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding='utf-8')
analysis_summary


{'schema_version': 'graph_adapter_batch_review.v1',
 'status': 'completed',
 'train_rows': 82,
 'test_rows': 26,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level',
  'vix_change_1d',
  'intraday_range'],
 'models': ['market_graph_mse',
  'gcn_mse',
  'gat_mse',
  'rgcn_mse',
  'rsr_mse',
  'hypergraph_mse',
  'thgnn_mse',
  'sthan_mse',
  'sthgcn_mse'],
 'adapter_statuses': ['paper_strict_adapter'],
 'model_adapters': ['gat_feature_attention_mse_v1',
  'gcn_feature_propagation_mse_v1',
  'hypergraph_feature_mse_v1',
  'market_graph_feature_mse_v1',
  'rgcn_relation_feature_mse_v1',
  'rsr_relation_strength_mse_v1',
  'sthan_spatio_temporal_attention_mse_v1',
  'sthgcn_spatio_temporal_hypergraph_mse_v1',
  'thgnn_temporal_hypergraph_mse_v1'],
 'gat_vs_gcn_abs_mean_diff': 0.01857881705538094,
 'rgcn_vs_gcn_abs_mean_diff': 0.00010161405019317535,
 'hypergraph_vs_gcn_abs_mean_diff': 0.0134195732081497

## 中文结论

- 做了什么：新增并验证 `market_graph_mse`、`gcn_mse`、`gat_mse`、`rgcn_mse`、`rsr_mse`、`hypergraph_mse`、`thgnn_mse`、`sthan_mse`、`sthgcn_mse` 九个 graph/network adapter；当前实现是 graph propagation、attention、relation、hypergraph 和 temporal graph feature proxy。
- 核心指标：九个模型的 adapter 状态、预测均值/波动、预测与目标相关性写入 `adapter_summary.csv`；GAT vs GCN、R-GCN vs GCN、Hypergraph vs GCN、THGNN vs Hypergraph 的平均绝对预测差异写入 `analysis_summary.json`。
- regime 差异：本轮小矩阵通过 `matrix_card` 写出 `regime_metrics.csv`、`window_metrics.csv`、`qlib_risk_metrics.csv` 和 `model_effect_deltas.csv`，证明 graph adapter 能进入已有 qlib/regime 分析链路。
- 是否晋升：否。本轮是 adapter 框架推进，不代表生产策略晋升。
- 风险：当前均为 feature proxy，不是完整 GNN/Hypergraph/Spatio-temporal 论文级实现；后续要在同一 adapter/schema 下替换为更严格图神经网络实现，并显式引入真实资产关系图。
- 下一步：补齐剩余 `mlp_mse` 严格 adapter、完善 objective/promotion 评分，并做一次完整 QuantBench 模型覆盖审计。